# TravelMind : Version rattrapage

Cette version recentre le projet sur une lecture plus claire et plus convergente suite aux retours jury.

Objectif initial : prédire avant le départ un **score de satisfaction client de 1 à 5** à partir des seules informations disponibles en pré-voyage.

Suite aux dernières expériences, la formulation la plus exploitable pour le rattrapage est une **classification binaire pré-voyage** :

- `0` : satisfaction faible ou moyenne (`1`, `2`, `3`) ;
- `1` : satisfaction haute (`4`, `5`).

Le meilleur candidat actuel est `LogisticRegression_balanced_optimized`, entraîné sur la version expérimentale `v2.2-minimal_balanced`.

Message central : le modèle devient exploitable comme **prototype industrialisable**, avec `F1 = 0.5995`, `ROC AUC = 0.7320` et `balanced accuracy = 0.7057`. Il reste à valider sur données réelles, car une partie du dataset `v2.2` est synthétique.


## 1. Cadrage synthétique

### Problème métier

Une agence de voyages haut de gamme souhaite identifier avant le départ les séjours susceptibles de générer une satisfaction faible ou moyenne, afin de renforcer l'accompagnement humain.

### Plus-value attendue

- Donner au conseiller un score indicatif de satisfaction probable.
- Prioriser les dossiers nécessitant une revue humaine.
- Structurer un pipeline reproductible pour préparer une future amélioration avec données réelles.

### Limite assumée

La prédiction ne remplace pas le conseiller. Avec les données actuelles, elle sert principalement de prototype analytique et industrialisable.


## 2. Imports et configuration


In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.model_selection import train_test_split

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from app.modeling import (
    RANDOM_STATE,
    TARGET_COLUMN,
    add_base_features,
    clean_dataset,
    prepare_training_dataset,
    train_and_select_model,
)

ENRICHED_DATA_PATH = PROJECT_ROOT / "data" / "versions" / "v2_1_enrichment" / "travel_planning_dataset_v2_1.csv"
RAW_DATA_PATH = PROJECT_ROOT / "data" / "Examen_travel_planning_dataset.csv"
DATA_PATH = ENRICHED_DATA_PATH if ENRICHED_DATA_PATH.exists() else RAW_DATA_PATH
MODEL_PATH = PROJECT_ROOT / "models" / "model_pre_voyage.pkl"
METADATA_PATH = PROJECT_ROOT / "models" / "model_pre_voyage_metadata.json"

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_rows", 30)
sns.set_theme(style="whitegrid")


## 3. Chargement du dataset

Le dataset utilisé est synthétique, anonymisé et fourni dans le cadre du projet. Il contient des séjours passés avec leurs caractéristiques, contraintes, résultats opérationnels et satisfaction client.


### 3.1 Dataset enrichi utilise pour le rattrapage

Le notebook utilise maintenant en priorite le dataset `data/versions/v2_1_enrichment/travel_planning_dataset_v2_1.csv`.

Cette version ajoute deux enrichissements reels issus d'Open-Meteo, joints par `destination` et `saison` : meteo historique et qualite de l'air historique.

Si ce fichier n'existe pas, le notebook revient automatiquement au dataset brut `data/Examen_travel_planning_dataset.csv`.

Les colonnes tarifaires ne sont pas inventees : elles ne seront ajoutees que si une source tarifaire reelle et autorisee est fournie dans `data/external/tariff_market_context.csv`.


In [ ]:
df_raw = pd.read_csv(DATA_PATH)
print(f"Dataset utilise : {DATA_PATH.relative_to(PROJECT_ROOT)}")
print(f"Nombre de lignes : {df_raw.shape[0]}")
print(f"Nombre de colonnes : {df_raw.shape[1]}")
display(df_raw.head())


In [ ]:
datasheet_variables = pd.DataFrame([
    {"variable": "trip_id", "role": "identifiant", "usage": "Exclu du modèle"},
    {"variable": "client_type", "role": "profil client", "usage": "Entrée pré-voyage"},
    {"variable": "budget_total", "role": "budget global", "usage": "Entrée pré-voyage"},
    {"variable": "destination", "role": "destination", "usage": "Entrée pré-voyage"},
    {"variable": "saison", "role": "période", "usage": "Entrée pré-voyage"},
    {"variable": "duree_jours", "role": "durée", "usage": "Entrée pré-voyage"},
    {"variable": "type_hebergement", "role": "hébergement", "usage": "Entrée pré-voyage"},
    {"variable": "prix_vol", "role": "coût transport", "usage": "Entrée pré-voyage"},
    {"variable": "meteo_prevue", "role": "contexte météo", "usage": "Entrée pré-voyage"},
    {"variable": "activite_principale", "role": "centre d'intérêt", "usage": "Entrée pré-voyage"},
    {"variable": "satisfaction_client", "role": "cible", "usage": "Score à prédire"},
    {"variable": "imprevus", "role": "événement post-voyage", "usage": "Exclu du modèle pré-voyage"},
    {"variable": "reorganisation_necessaire", "role": "résultat opérationnel", "usage": "Exclu du modèle pré-voyage"},
    {"variable": "respect_budget", "role": "résultat budgétaire", "usage": "Exclu du modèle pré-voyage"},
    {"variable": "retour_client", "role": "avis post-voyage", "usage": "Exclu du modèle pré-voyage"},
])
display(datasheet_variables)


## 4. Préparation des données - synthèse C3

### 4.1 Finalité

La préparation des données vise un objectif précis : construire un dataset **cohérent, traçable et exploitable pour un modèle pré-voyage**.

La plus-value n'est pas de multiplier les traitements, mais de sécuriser les données utilisées par le modèle :

- conserver uniquement les informations disponibles avant le départ ;
- supprimer les cas impossibles ou contradictoires pour éviter un apprentissage incohérent ;
- centraliser les règles métier dans `configs/business_rules.json` ;
- laisser les traitements apprenants dans le pipeline `scikit-learn` après le split train/test.

### 4.2 Principe de décision

Seules les incohérences critiques sont supprimées. Les cas discutables mais métier possibles sont conservés et documentés, afin de ne pas appauvrir artificiellement le dataset.


In [ ]:
from app.config import load_business_rules

# Lecture des règles métier centralisées.
# Elles évitent de modifier le notebook à chaque évolution de borne, catégorie ou seuil.
business_rules = load_business_rules()

contraintes_numeriques = pd.DataFrame([
    {
        "variable": variable,
        "minimum": limites.get("min"),
        "maximum": limites.get("max"),
        "objectif": {
            "duree_jours": "écarter les durées irréalistes",
            "budget_total": "écarter les budgets hors périmètre métier",
            "prix_vol": "écarter les coûts de vol hors périmètre métier",
        }.get(variable, "contrôle métier"),
    }
    for variable, limites in business_rules["api_constraints"].items()
])

categories_fermees = pd.DataFrame([
    {
        "variable": variable,
        "valeurs_autorisees": ", ".join(valeurs),
        "objectif": "éviter les catégories inconnues sur les champs métier fermés",
    }
    for variable, valeurs in business_rules["allowed_categories"].items()
])

display(contraintes_numeriques)
display(categories_fermees)


In [ ]:
# Contrôles d'intégrité initiaux.
# Ces contrôles servent à distinguer les anomalies critiques des simples cas métier atypiques.
imprevus_norm = (
    df_raw["imprevus"]
    .fillna("aucun")
    .astype("string")
    .str.strip()
    .str.lower()
    .replace({"": "aucun", "nan": "aucun"})
)

def compter_hors_bornes(variable):
    limites = business_rules["api_constraints"][variable]
    valeurs = pd.to_numeric(df_raw[variable], errors="coerce")
    masque = pd.Series(False, index=df_raw.index)
    if limites.get("min") is not None:
        masque |= valeurs < limites["min"]
    if limites.get("max") is not None:
        masque |= valeurs > limites["max"]
    return masque

masque_hors_bornes = pd.Series(False, index=df_raw.index)
for variable in business_rules["api_constraints"]:
    masque_hors_bornes |= compter_hors_bornes(variable)

controle_qualite_initial = pd.DataFrame([
    {
        "controle": "doublons trip_id",
        "nb_lignes": int(df_raw["trip_id"].duplicated().sum()),
        "decision": "supprimer si présent",
        "plus_value": "éviter le double comptage d'un séjour",
    },
    {
        "controle": "satisfaction absente ou hors échelle 1-5",
        "nb_lignes": int((df_raw[TARGET_COLUMN].isna() | ~df_raw[TARGET_COLUMN].between(1, 5)).sum()),
        "decision": "supprimer",
        "plus_value": "garantir une cible d'apprentissage valide",
    },
    {
        "controle": "valeurs hors bornes métier configurées",
        "nb_lignes": int(masque_hors_bornes.sum()),
        "decision": "supprimer",
        "plus_value": "écarter les séjours hors périmètre du cas d'usage",
    },
    {
        "controle": "prix_vol > budget_total",
        "nb_lignes": int((df_raw["prix_vol"] > df_raw["budget_total"]).sum()),
        "decision": "supprimer",
        "plus_value": "éviter une incohérence budgétaire forte",
    },
    {
        "controle": "aucun imprévu mais réorganisation nécessaire",
        "nb_lignes": int(((imprevus_norm == "aucun") & (df_raw["reorganisation_necessaire"] == 1)).sum()),
        "decision": "supprimer",
        "plus_value": "éviter une contradiction opérationnelle forte",
    },
])

display(controle_qualite_initial)

# Application du nettoyage stabilisé utilisé aussi par train.py.
df_clean, cleaning_report = clean_dataset(df_raw)
df_model = add_base_features(df_clean)

finalites_preparation = {
    "dataset_brut": "point de départ fourni",
    "cible_satisfaction_client_valide": "cible exploitable pour la régression",
    "contraintes_metier_config": "périmètre métier cohérent avec l'API",
    "coherence_initiale_prix_vol_budget_total": "budget total compatible avec le coût du vol",
    "reorganisation_sans_imprevu_declare": "cohérence opérationnelle minimale",
}

rapport_nettoyage = pd.DataFrame(cleaning_report)
rapport_nettoyage["lignes_supprimees_cumulees"] = len(df_raw) - rapport_nettoyage["nb_lignes"]
rapport_nettoyage["lignes_restantes_pct"] = (rapport_nettoyage["nb_lignes"] / len(df_raw) * 100).round(2)
rapport_nettoyage["finalite"] = rapport_nettoyage["etape"].map(finalites_preparation)

display(rapport_nettoyage)

print(f"Lignes initiales : {len(df_raw)}")
print(f"Lignes après nettoyage : {len(df_clean)}")
print(f"Lignes supprimées : {len(df_raw) - len(df_clean)}")


### 4.3 Traitements placés dans le pipeline

Les traitements qui apprennent des paramètres ne sont pas appliqués directement sur tout le dataset avant la séparation train/test.

Ils sont intégrés au pipeline `scikit-learn` :

- imputation des valeurs manquantes ;
- traitement IQR des valeurs aberrantes numériques ;
- standardisation des variables numériques continues ;
- encodage `OneHotEncoder` des variables catégorielles.

Cette organisation limite le risque de fuite de données entre le train et le test.

### 4.4 Conclusion C3 - plus-value de la préparation

La préparation réduit le dataset de `1500` à `1378` lignes, uniquement après suppression de cas invalides, hors périmètre ou fortement incohérents. Le dataset final est plus fiable, mieux aligné avec l'usage pré-voyage et réutilisable par le notebook, `train.py`, l'API et le monitoring.

La section C3 est donc recentée sur la finalité : **produire une base propre et défendable**, sans prêtendre améliorer artificiellement la performance du modèle.


## 5. Feature engineering retenu

Les nouvelles variables sont limitées à des informations disponibles avant le départ.


In [ ]:
x_pre, y_pre, cleaning_report = prepare_training_dataset(df_raw)

features_resume = pd.DataFrame([
    {"feature": "budget_par_jour", "definition": "budget_total / duree_jours", "raison": "Comparer les séjours de durées différentes"},
    {"feature": "part_vol_budget", "definition": "prix_vol / budget_total", "raison": "Mesurer le poids du transport dans le budget"},
    {"feature": "sejour_long", "definition": "1 si duree_jours >= 14", "raison": "Identifier les séjours longs"},
    {"feature": "meteo_risque", "definition": "1 si pluie ou variable", "raison": "Repérer un contexte météo moins favorable"},
    {"feature": "client_business", "definition": "1 si client_type = business", "raison": "Isoler les voyages professionnels"},
    {"feature": "hebergement_luxe", "definition": "1 si resort ou villa", "raison": "Identifier un niveau d'hébergement premium"},
])

display(features_resume)
print(f"Nombre total de variables d'entrée : {x_pre.shape[1]}")
display(pd.DataFrame({"features_modelisation": x_pre.columns}))


## 6. Variables exclues pour éviter la fuite de données

Les variables post-voyage sont volontairement exclues du modèle pré-voyage : elles ne sont pas connues au moment où l'agence veut anticiper la satisfaction.


In [ ]:
variables_exclues = pd.DataFrame([
    {"variable": "imprevus", "raison": "Connu pendant ou après le séjour"},
    {"variable": "reorganisation_necessaire", "raison": "Conséquence opérationnelle post-voyage"},
    {"variable": "respect_budget", "raison": "Résultat constaté après séjour"},
    {"variable": "retour_client", "raison": "Avis client post-voyage, trop proche de la cible"},
    {"variable": "trip_id", "raison": "Identifiant sans valeur prédictive métier"},
])

display(variables_exclues)


## 7. Choix du modèle IA - synthèse C4

### 7.1 Logique de convergence

Suite aux retours jury, le choix du modèle est présenté comme un **entonnoir de décision** : on part du besoin métier, on teste quelques familles représentatives, puis on retient le meilleur compromis.

### Besoin métier traité

Le besoin retenu pour la version rattrapage est de prédire **avant le départ** un score indicatif de satisfaction client sur l'échelle `1 à 5`.

Ce choix conduit à un problème de **régression supervisée** :

- entrée : variables connues avant le voyage ;
- sortie : score continu de satisfaction ;
- métriques principales : `MAE`, `RMSE`, `R2` ;
- métriques non prioritaires : rappel, matrice de confusion et ROC, car elles concernent surtout la classification.

### Critères de choix

Le modèle retenu doit être :

- adapté au cas d'usage pré-voyage ;
- comparable à une baseline naïve ;
- stable avec les variables catégorielles encodées ;
- explicable devant le métier ;
- sobre en calcul ;
- facilement industrialisable dans l'API, Docker et la CI/CD ;
- sans fuite de données post-voyage.


In [ ]:
# Entraînement et comparaison de modèles candidats.
# La sélection se fait sur la MAE : plus elle est basse, plus l'erreur moyenne est faible.
training_result = train_and_select_model(
    x=x_pre,
    y=y_pre,
    cleaning_report=cleaning_report,
    test_size=0.2,
)

resultats_modeles = pd.DataFrame(training_result.evaluation_results)
baseline_mae = resultats_modeles.loc[
    resultats_modeles["modele"] == "Dummy_mean_regression",
    "mae",
].iloc[0]
resultats_modeles["gain_mae_vs_baseline"] = baseline_mae - resultats_modeles["mae"]

criteres_modeles = pd.DataFrame([
    {
        "modele": "Dummy_mean_regression",
        "famille": "baseline naïve",
        "role": "point de comparaison minimal",
        "decision": "non retenu : ne modélise aucune relation",
    },
    {
        "modele": "LinearRegression_pre",
        "famille": "régression linéaire simple",
        "role": "modèle interprétable de référence",
        "decision": "non retenu : moins stable avec l'encodage catégoriel",
    },
    {
        "modele": "RidgeRegression_pre",
        "famille": "régression linéaire régularisée",
        "role": "compromis performance, stabilité et explicabilité",
        "decision": "non retenu après nettoyage strict : ne dépasse plus la baseline",
    },
    {
        "modele": "RandomForestRegressor_pre",
        "famille": "ensemble non linéaire",
        "role": "tester si des relations non linéaires améliorent le score",
        "decision": "non retenu : complexité supérieure sans gain suffisant",
    },
])

synthese_choix = (
    resultats_modeles
    .merge(criteres_modeles, on="modele", how="left")
    .sort_values(["mae", "rmse"], ascending=[True, True])
    .reset_index(drop=True)
)

colonnes_synthese = [
    "modele",
    "famille",
    "mae",
    "rmse",
    "r2",
    "gain_mae_vs_baseline",
    "role",
    "decision",
]

display(synthese_choix[colonnes_synthese].round(4))

print(f"Modèle retenu : {training_result.model_name}")
print(json.dumps(training_result.metrics, indent=2, ensure_ascii=False))


### 7.2 Lecture des métriques et décision

- `MAE` : erreur moyenne absolue en points de satisfaction. Une MAE d'environ `1.06` signifie que le modèle se trompe en moyenne d'environ un point sur l'échelle `1 à 5`.
- `RMSE` : mesure proche de la MAE, mais plus sensible aux grosses erreurs.
- `R2` : part de variance expliquée par le modèle. Une valeur proche de `0` indique que les variables pré-voyage expliquent très peu la satisfaction finale.
- `gain_mae_vs_baseline` : écart entre le modèle testé et la baseline `Dummy_mean_regression`. Un gain nul ou négatif signifie que le modèle n'apprend pas mieux qu'une prédiction moyenne.

### Lecture chiffrée des résultats actuels

- Après nettoyage strict, `Dummy_mean_regression` obtient la meilleure `MAE` : environ `1.0636`.
- `RandomForestRegressor_pre` obtient `MAE = 1.0686`, donc fait légèrement moins bien que la baseline.
- `RidgeRegression_pre` obtient `MAE = 1.0742` et `R2 = -0.0356`, ce qui confirme l'absence de signal pré-voyage suffisant.
- Aucun modèle candidat ne produit une performance suffisante pour une aide à la décision fiable.

### 7.3 Décision C4

Le nettoyage strict améliore la cohérence du dataset, mais il confirme aussi une limite forte : les variables pré-voyage disponibles ne permettent pas de prédire correctement la satisfaction client.

La baseline devient le meilleur résultat statistique. Elle ne doit pas être présentée comme une solution IA métier, car elle prédit essentiellement la moyenne historique. La conclusion retenue est donc : **prototype industrialisable techniquement, mais non recommandé pour une mise en production décisionnelle sans enrichissement de données réelles pré-voyage**.

Les pistes plus complexes, notamment Random Forest, classification, SMOTE, Optuna ou NLP, sont écartées du modèle principal car elles ajoutent de la complexité sans résoudre le manque de signal pré-voyage ou introduisent des variables post-voyage incompatibles avec l'objectif avant départ.


## 8. Entraînement supervisé et diagnostic - synthèse C5

### 8.1 Protocole d'entraînement retenu

Le modèle est entraîné de façon automatique et supervisée : la cible `satisfaction_client` est connue dans l'historique et sert à apprendre une relation avec les variables pré-voyage.

Pour répondre à la remarque du jury sur le risque de biais lié au prétraitement, le protocole retenu impose que les traitements qui apprennent des paramètres soient placés dans le pipeline `scikit-learn`. Ils sont donc appris uniquement sur le jeu d'entraînement, puis appliqués au jeu de test.

### 8.2 Réponse au point recall / matrice de confusion

La version rattrapage retient une régression. Les métriques principales ne sont donc plus le `recall`, la matrice de confusion ou la courbe ROC, mais `MAE`, `RMSE` et `R2`.

Pour garder une lecture exploitable par le jury, un diagnostic secondaire est ajouté : les scores réels et prédits sont regroupés en trois zones (`insatisfait_1_2`, `neutre_3`, `satisfait_4_5`). Ce diagnostic produit un rappel indicatif par zone et une matrice de confusion lisible, mais il ne sert pas à choisir le modèle.


In [ ]:
protocole_entrainement = pd.DataFrame([
    {
        "etape": "Séparation train/test",
        "methode": "train_test_split avec stratification sur le score entier",
        "raison": "conserver une répartition comparable des scores entre train et test",
    },
    {
        "etape": "Imputation",
        "methode": "SimpleImputer dans le pipeline",
        "raison": "apprendre les valeurs de remplacement uniquement sur le train",
    },
    {
        "etape": "Outliers",
        "methode": "IQRMedianOutlierReplacer dans le pipeline",
        "raison": "traiter les valeurs aberrantes sans fuite train/test",
    },
    {
        "etape": "Standardisation",
        "methode": "StandardScaler dans le pipeline",
        "raison": "mettre les variables numériques continues sur une échelle comparable",
    },
    {
        "etape": "Encodage",
        "methode": "OneHotEncoder(handle_unknown='ignore') dans le pipeline",
        "raison": "encoder les catégories sans imposer d'ordre artificiel",
    },
    {
        "etape": "Variables exclues",
        "methode": "suppression des variables post-voyage",
        "raison": "éviter la fuite de données",
    },
])

display(protocole_entrainement)


In [ ]:
# Diagnostic d'erreurs sur le jeu de test.
# On reconstruit le même split que dans train_and_select_model pour analyser les erreurs.
x_train, x_test, y_train, y_test = train_test_split(
    x_pre,
    y_pre,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_pre.astype(int),
)

predictions = np.clip(training_result.pipeline.predict(x_test), 1, 5)
residus = y_test.to_numpy() - predictions

evaluation_detaillee = pd.DataFrame({
    "satisfaction_reelle": y_test.to_numpy(),
    "satisfaction_predite": predictions,
    "erreur": residus,
    "erreur_absolue": np.abs(residus),
})

erreurs_par_score = (
    evaluation_detaillee
    .groupby("satisfaction_reelle")
    .agg(
        nb_lignes=("erreur_absolue", "size"),
        mae=("erreur_absolue", "mean"),
        prediction_moyenne=("satisfaction_predite", "mean"),
    )
    .reset_index()
    .round(4)
)

display(erreurs_par_score)


In [ ]:
# Diagnostic secondaire pour répondre à la remarque sur le recall et la matrice de confusion.
# Attention : le modèle reste une régression ; ce regroupement sert uniquement à rendre les erreurs lisibles.
def convertir_score_en_zone(scores):
    scores_arrondis = np.rint(np.clip(scores, 1, 5)).astype(int)
    return np.select(
        [scores_arrondis <= 2, scores_arrondis == 3, scores_arrondis >= 4],
        [0, 1, 2],
    )

libelles_zones = {
    0: "insatisfait_1_2",
    1: "neutre_3",
    2: "satisfait_4_5",
}
ordre_zones = [
    "insatisfait_1_2",
    "neutre_3",
    "satisfait_4_5",
]

y_test_zones = convertir_score_en_zone(y_test.to_numpy())
predictions_zones = convertir_score_en_zone(predictions)

diagnostic_zones = pd.DataFrame({
    "zone_reelle": [libelles_zones[int(zone)] for zone in y_test_zones],
    "zone_predite": [libelles_zones[int(zone)] for zone in predictions_zones],
})

recall_indicatif = (
    diagnostic_zones
    .assign(prediction_correcte=lambda df: df["zone_reelle"] == df["zone_predite"])
    .groupby("zone_reelle")
    .agg(
        nb_lignes=("prediction_correcte", "size"),
        bonnes_predictions=("prediction_correcte", "sum"),
        recall_indicatif=("prediction_correcte", "mean"),
    )
    .reindex(ordre_zones)
    .reset_index()
)
recall_indicatif["recall_indicatif"] = recall_indicatif["recall_indicatif"].round(4)

matrice_confusion_indicative = pd.crosstab(
    diagnostic_zones["zone_reelle"],
    diagnostic_zones["zone_predite"],
    rownames=["réel"],
    colnames=["prédit"],
)
matrice_confusion_indicative = matrice_confusion_indicative.reindex(
    index=ordre_zones,
    columns=ordre_zones,
    fill_value=0,
)

display(recall_indicatif)
display(matrice_confusion_indicative)


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sns.scatterplot(
    data=evaluation_detaillee,
    x="satisfaction_reelle",
    y="satisfaction_predite",
    alpha=0.6,
    ax=axes[0],
)
axes[0].plot([1, 5], [1, 5], color="red", linestyle="--", label="prédiction parfaite")
axes[0].set_title("Score réel vs score prédit")
axes[0].set_xlim(0.8, 5.2)
axes[0].set_ylim(0.8, 5.2)
axes[0].legend()

sns.histplot(evaluation_detaillee["erreur"], bins=20, kde=True, ax=axes[1])
axes[1].axvline(0, color="red", linestyle="--")
axes[1].set_title("Distribution des erreurs")

plt.tight_layout()
plt.show()


### 8.3 Conclusion C5

Le modèle est bien entraîné automatiquement et supervisé, avec un protocole plus robuste que la version initiale : split train/test avant prétraitement, pipeline complet, exclusion des variables post-voyage et comparaison à une baseline.

Le diagnostic secondaire rend la matrice de confusion exploitable : il montre que le modèle prédit majoritairement la zone `neutre_3`. Sur l'exécution actuelle, le rappel indicatif est d'environ `0.13` pour `insatisfait_1_2`, `0.91` pour `neutre_3` et `0.00` pour `satisfait_4_5`.

Conclusion : C5 valide la robustesse du processus d'entraînement, mais ne valide pas une performance métier suffisante. La cause principale n'est pas le choix d'algorithme, mais le manque de signal dans les variables pré-voyage disponibles. Le modèle ne doit donc pas être utilisé comme aide à la décision en production.


## 9. Architecture cible industrialisée - synthèse C7

### 9.1 Besoin d'architecture

L'architecture doit permettre de sortir du notebook et de rendre le prototype testable dans un environnement technique cohérent.

Objectifs couverts :

- entraîner le modèle pré-voyage de façon reproductible ;
- exposer une prédiction via une API ;
- proposer une interface simple de test ;
- journaliser les prédictions ;
- surveiller les dérives ;
- valider automatiquement le projet avec la CI/CD ;
- conserver une architecture proportionnée à un prototype.

### 9.2 Contraintes prises en compte

- Performance modèle faible : le système ne doit pas prendre de décision automatique.
- Données synthétiques : la production nécessite une validation sur données réelles.
- Reproductibilité : l'entraînement doit pouvoir être rejoué hors notebook.
- Sobriété : modèle tabulaire simple, pas de NLP lourd en production.
- Portabilité : API conteneurisable avec Docker.
- Gouvernance : validation métier, DSI et juridique requise avant généralisation.

### 9.3 Architecture retenue

Le flux logique retenu est :

`Dataset brut` -> `train.py` -> `artefacts modèle` -> `API FastAPI` -> `Streamlit` -> `logs et monitoring` -> `CI/CD et Docker`.

Cette architecture est adaptée à un **prototype industrialisable**, pas à une production client autonome.


In [ ]:
architecture_briques = pd.DataFrame([
    {
        "ordre": 1,
        "brique": "Données",
        "fichiers": "data/Examen_travel_planning_dataset.csv, configs/business_rules.json",
        "role": "source d'entraînement et règles métier centralisées",
        "contrainte_couverte": "traçabilité et cohérence métier",
        "statut": "mis en oeuvre",
    },
    {
        "ordre": 2,
        "brique": "Entraînement",
        "fichiers": "train.py, app/modeling.py",
        "role": "rejouer le nettoyage, le feature engineering et l'entraînement",
        "contrainte_couverte": "reproductibilité hors notebook",
        "statut": "mis en oeuvre",
    },
    {
        "ordre": 3,
        "brique": "Artefacts modèle",
        "fichiers": "models/model_pre_voyage.pkl, models/model_pre_voyage_metadata.json",
        "role": "stocker le pipeline entraîné, les métriques et le profil de référence",
        "contrainte_couverte": "traçabilité des performances",
        "statut": "mis en oeuvre",
    },
    {
        "ordre": 4,
        "brique": "API",
        "fichiers": "app/main.py, app/predictor.py, app/schemas.py",
        "role": "servir /health, /predict et les endpoints de monitoring",
        "contrainte_couverte": "intégration technique et validation des entrées",
        "statut": "mis en oeuvre",
    },
    {
        "ordre": 5,
        "brique": "Interface de test",
        "fichiers": "app_web.py",
        "role": "tester la prédiction sans requête API manuelle",
        "contrainte_couverte": "accessibilité pour le métier",
        "statut": "mis en oeuvre",
    },
    {
        "ordre": 6,
        "brique": "Monitoring",
        "fichiers": "app/monitoring.py, logs/predictions/predictions.jsonl",
        "role": "journaliser les prédictions, suivre les zones d'incertitude et le data drift",
        "contrainte_couverte": "surveillance et revue humaine",
        "statut": "mis en oeuvre",
    },
    {
        "ordre": 7,
        "brique": "CI/CD et conteneurisation",
        "fichiers": ".github/workflows/ci-cd.yml, Dockerfile, docker-compose.yml",
        "role": "tester, contrôler la quality gate et construire l'image Docker si nécessaire",
        "contrainte_couverte": "non-régression et portabilité",
        "statut": "mis en oeuvre sans déploiement serveur automatique",
    },
])

scenarios_architecture = pd.DataFrame([
    {
        "scenario": "Notebook seul",
        "avantage": "simple pour l'analyse",
        "limite": "pas exploitable par une application",
        "decision": "non retenu seul",
    },
    {
        "scenario": "Script + API locale",
        "avantage": "modèle rejouable et prédiction testable",
        "limite": "usage local uniquement",
        "decision": "retenu pour le prototype",
    },
    {
        "scenario": "Docker local",
        "avantage": "environnement portable",
        "limite": "ne suffit pas à une production distante",
        "decision": "retenu pour la portabilité",
    },
    {
        "scenario": "VPS ou cloud managé",
        "avantage": "accès distant et supervision avancée",
        "limite": "coût, sécurité et validation DSI nécessaires",
        "decision": "non retenu à ce stade",
    },
])

acteurs_a_consulter = pd.DataFrame([
    {
        "acteur": "Commanditaire métier",
        "point_a_valider": "usage autorisé du score et seuils d'action humaine",
        "statut": "à consulter avant généralisation",
    },
    {
        "acteur": "DSI",
        "point_a_valider": "hébergement, sécurité, logs, sauvegardes et disponibilité",
        "statut": "à consulter avant déploiement distant",
    },
    {
        "acteur": "DPO / juridique",
        "point_a_valider": "RGPD, AI Act, durée de conservation et information utilisateur",
        "statut": "à consulter avant données réelles",
    },
])

display(architecture_briques)
display(scenarios_architecture)
display(acteurs_a_consulter)


## 10. API, monitoring et endpoints

Endpoints disponibles :

- `GET /health` : vérifier que l'API fonctionne ;
- `POST /predict` : prédire le score de satisfaction pré-voyage ;
- `GET /monitoring/summary` : résumer les prédictions journalisées ;
- `GET /monitoring/drift` : comparer les nouvelles entrées au profil d'entraînement ;
- `GET /monitoring/alerts` : synthétiser les alertes et proposer une action.

Le monitoring suit les entrées API, les scores prédits, les zones d'incertitude et les dérives. Il ne mesure pas encore la vraie performance en production, car cela nécessiterait des satisfactions réelles collectées après séjour.


In [ ]:
if METADATA_PATH.exists():
    metadata = json.loads(METADATA_PATH.read_text(encoding="utf-8"))
    display(pd.DataFrame([metadata["metrics"]]).round(4))
    print(f"Objectif industrialisé : {metadata.get('objective')}")
    print(f"Modèle industrialisé : {metadata.get('model_name')}")
else:
    print("Artefacts modèle absents. Lancer : python train.py")


## 11. Mesure de performance et impacts - synthèse C8

### 11.1 Objectif de la mesure

La mesure de performance ne sert pas seulement à afficher des scores. Elle doit permettre de décider si la solution peut être utilisée, sous quelles limites et avec quelles actions de suivi.

Dans TravelMind, l'objectif est donc double :

- mesurer si le modèle pré-voyage apporte une vraie valeur par rapport à une baseline ;
- mesurer si son exploitation est acceptable au regard des impacts métier, éthiques et environnementaux.

### 11.2 Indicateurs retenus

- `MAE` : erreur moyenne en points de satisfaction sur l'échelle `1 à 5`.
- `RMSE` : erreur qui pénalise davantage les grosses erreurs.
- `R2` : capacité du modèle à expliquer la satisfaction.
- `gain_mae_vs_baseline` : valeur ajoutée par rapport à une prédiction naïve.
- `zone_incertitude` : indicateur d'exploitation pour imposer une revue humaine si le score est intermédiaire.
- `data drift` : écart entre les données reçues par l'API et le profil d'entraînement.
- `empreinte carbone` : impact environnemental de l'entraînement mesuré avec CodeCarbon lorsque disponible.


In [ ]:
# Synthèse C8 : transformer les métriques en décision d'exploitation.
metrics_c8 = training_result.metrics

performance_c8 = pd.DataFrame([
    {
        "indicateur": "MAE",
        "valeur": round(metrics_c8["mae"], 4),
        "lecture": "erreur moyenne d'environ 1 point sur une échelle 1-5",
        "conclusion": "performance faible pour une aide à la décision",
    },
    {
        "indicateur": "RMSE",
        "valeur": round(metrics_c8["rmse"], 4),
        "lecture": "les grosses erreurs restent significatives",
        "conclusion": "prudence nécessaire sur les cas individuels",
    },
    {
        "indicateur": "R2",
        "valeur": round(metrics_c8["r2"], 4),
        "lecture": "variance de satisfaction presque non expliquée",
        "conclusion": "signal pré-voyage insuffisant",
    },
    {
        "indicateur": "gain MAE vs baseline",
        "valeur": round(metrics_c8["mae_gain_vs_baseline"], 4),
        "lecture": "gain très limité face à une prédiction moyenne",
        "conclusion": "modèle non convaincant en performance pure",
    },
])

decision_exploitation = pd.DataFrame([
    {
        "usage": "Démonstrateur technique",
        "decision": "autorisé",
        "raison": "le pipeline, l'API, le monitoring et la CI/CD sont testables",
    },
    {
        "usage": "Support d'analyse interne",
        "decision": "autorisé avec prudence",
        "raison": "le score peut aider à discuter les limites des données",
    },
    {
        "usage": "Aide à la décision commerciale",
        "decision": "non autorisé en l'état",
        "raison": "performance trop proche de la baseline",
    },
    {
        "usage": "Décision automatique client",
        "decision": "interdit",
        "raison": "risque de surinterprétation et absence de fiabilité suffisante",
    },
])

display(performance_c8)
display(decision_exploitation)


In [ ]:
# Bilan carbone : lecture de la dernière mesure CodeCarbon si elle existe.
# Si le fichier n'existe pas sur un autre poste, la cellule reste non bloquante.
codecarbon_path = PROJECT_ROOT / "logs" / "codecarbon" / "emissions_notebook_final.csv"

if codecarbon_path.exists():
    emissions_df = pd.read_csv(codecarbon_path)
    derniere_mesure_carbone = emissions_df.tail(1).copy()
    derniere_mesure_carbone["emissions_g_co2e"] = derniere_mesure_carbone["emissions"] * 1000
    colonnes_carbone = [
        "timestamp",
        "duration",
        "emissions",
        "emissions_g_co2e",
        "energy_consumed",
        "cpu_power",
        "ram_power",
        "country_iso_code",
    ]
    display(derniere_mesure_carbone[colonnes_carbone].round(8))
else:
    print("Mesure CodeCarbon absente sur ce poste. Lancer une mesure carbone si besoin.")


### 11.3 Conclusion C8 - décision d'exploitation

Les métriques ne valident pas une mise en production comme outil d'aide à la décision. Elles valident uniquement la faisabilité technique d'un pipeline IA industrialisable.

Conclusion opérationnelle :

- le modèle peut être utilisé pour démontrer l'architecture et le processus MLOps ;
- le score prédit doit rester indicatif ;
- aucune décision automatique ne doit être prise à partir du score ;
- une revue humaine est obligatoire pour tout usage métier ;
- le passage en production exige des données réelles, une nouvelle validation de performance et une revue des biais.

### 11.4 Conclusion impact environnemental

L'impact carbone mesuré est faible car le modèle retenu est un modèle tabulaire simple et le NLP lourd n'est pas industrialisé. Cette sobriété est cohérente avec la faible valeur ajoutée observée : il ne serait pas justifié d'utiliser un modèle beaucoup plus lourd sans gain robuste.

### 11.5 Actions d'amélioration déclenchées

- maintenir le modèle en mode prototype ;
- afficher clairement les limites dans l'API et l'interface ;
- suivre les zones d'incertitude et le data drift ;
- collecter des données réelles pré-voyage plus explicatives ;
- comparer tout nouveau modèle à la baseline avant remplacement ;
- soumettre toute généralisation au métier, à la DSI et au DPO/juridique.


## 12. Éthique, RGPD et AI Act

Points de maîtrise :

- dataset actuel synthétique et anonymisé ;
- exclusion des variables post-voyage pour éviter la fuite de données ;
- supervision humaine obligatoire ;
- interdiction d'usage pour refuser automatiquement un client ou ajuster un prix individuellement ;
- transparence sur les limites et le faible niveau de performance ;
- si données réelles : information client, minimisation, sécurité, durée de conservation et validation DPO.

Positionnement AI Act : TravelMind est un outil d'aide à l'analyse pour une agence de voyages. Il n'est pas identifié comme système à haut risque dans le cadre actuel, mais il nécessite une alphabétisation IA des utilisateurs et une supervision humaine.


## 13. Amélioration continue

La priorité n'est pas d'ajouter des modèles plus complexes, mais d'améliorer les données disponibles avant le départ.

Données à collecter pour une version future :

- attentes explicites du client ;
- préférences détaillées ;
- contraintes personnelles ;
- historique client ;
- canal de réservation ;
- niveau d'accompagnement souhaité ;
- satisfaction réelle observée après séjour.

Le réentraînement devra être réalisé uniquement après validation métier, contrôle qualité et comparaison avec le modèle précédent.


## 14. Synthèse finale rattrapage

TravelMind démontre une démarche IA complète : cadrage, préparation des données, modélisation, industrialisation, API, interface, monitoring, Docker et CI/CD.

La version rattrapage clarifie la convergence :

- la régression pré-voyage sur la note exacte `1 à 5` reste peu performante sur le dataset initial ;
- l'enrichissement réel météo / qualité de l'air améliore le contexte mais ne suffit pas ;
- une version expérimentale `v2.2` à `3000` lignes, partiellement synthétique, permet de tester l'hypothèse d'un signal métier plus structuré ;
- la formulation la plus exploitable devient la **classification binaire** : satisfaction haute (`4-5`) vs satisfaction faible/moyenne (`1-3`).

Le meilleur candidat actuel est `LogisticRegression_balanced_optimized` sur `v2.2-minimal_balanced`.

Conclusion : le modèle est **industrialisable comme prototype contrôlé**, mais il ne doit pas être présenté comme prêt pour une production réelle tant qu'il n'a pas été validé sur des données réelles collectées par l'agence.


## 15. Experience complementaire - classification binaire equilibree

Cette section teste une formulation plus simple du probleme : predire si le client sera satisfait ou non.

La cible devient :

- `0 = satisfaction faible ou moyenne` pour les notes `1`, `2` et `3` ;
- `1 = satisfaction haute` pour les notes `4` et `5`.

L'objectif est de verifier si un modele de classification binaire capte mieux le signal que la regression sur la note exacte.

Pour traiter le desequilibre des classes, deux approches sont comparees :

1. `class_weight="balanced"` : le modele donne plus de poids a la classe minoritaire sans modifier les donnees ;
2. `SMOTE` : creation d'exemples synthetiques uniquement sur le jeu d'entrainement, apres pretraitement, afin d'eviter la fuite de donnees.

Cette experience reste exploratoire : elle ne remplace pas automatiquement le modele industrialise tant qu'elle n'est pas validee metier et techniquement.


In [ ]:
from sklearn.base import clone
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline

from app.modeling import build_preprocessor

try:
    from imblearn.over_sampling import SMOTE
    from imblearn.pipeline import Pipeline as ImbPipeline
    smote_disponible = True
except ImportError:
    smote_disponible = False

# Creation de la cible binaire : 1 = satisfaction haute, 0 = satisfaction faible ou moyenne.
y_binary = (y_pre >= 4).astype(int)

# Split stratifie : la proportion de clients satisfaits est conservee dans train et test.
x_train_bin, x_test_bin, y_train_bin, y_test_bin = train_test_split(
    x_pre,
    y_binary,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y_binary,
)

repartition_binaire = pd.DataFrame({
    "jeu": ["global", "train", "test"],
    "nb_lignes": [len(y_binary), len(y_train_bin), len(y_test_bin)],
    "taux_satisfaction_haute": [
        y_binary.mean(),
        y_train_bin.mean(),
        y_test_bin.mean(),
    ],
    "classe_0": [
        int((y_binary == 0).sum()),
        int((y_train_bin == 0).sum()),
        int((y_test_bin == 0).sum()),
    ],
    "classe_1": [
        int((y_binary == 1).sum()),
        int((y_train_bin == 1).sum()),
        int((y_test_bin == 1).sum()),
    ],
})

print("Distribution de la cible binaire")
display(repartition_binaire.round(4))

preprocess_binary, numeric_binary, categorical_binary = build_preprocessor(x_train_bin)

modeles_binaires = {
    "Dummy_majority_binary": Pipeline(steps=[
        ("preprocess", clone(preprocess_binary)),
        ("model", DummyClassifier(strategy="most_frequent")),
    ]),
    "LogisticRegression_balanced": Pipeline(steps=[
        ("preprocess", clone(preprocess_binary)),
        ("model", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=RANDOM_STATE)),
    ]),
    "RandomForest_balanced": Pipeline(steps=[
        ("preprocess", clone(preprocess_binary)),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=6,
            min_samples_leaf=10,
            class_weight="balanced",
            random_state=RANDOM_STATE,
            n_jobs=1,
        )),
    ]),
    "GradientBoosting_binary": Pipeline(steps=[
        ("preprocess", clone(preprocess_binary)),
        ("model", GradientBoostingClassifier(
            n_estimators=120,
            learning_rate=0.05,
            max_depth=2,
            random_state=RANDOM_STATE,
        )),
    ]),
}

if smote_disponible:
    modeles_binaires["LogisticRegression_SMOTE"] = ImbPipeline(steps=[
        ("preprocess", clone(preprocess_binary)),
        ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
        ("model", LogisticRegression(max_iter=1000, random_state=RANDOM_STATE)),
    ])
    modeles_binaires["RandomForest_SMOTE"] = ImbPipeline(steps=[
        ("preprocess", clone(preprocess_binary)),
        ("smote", SMOTE(random_state=RANDOM_STATE, k_neighbors=5)),
        ("model", RandomForestClassifier(
            n_estimators=200,
            max_depth=6,
            min_samples_leaf=10,
            random_state=RANDOM_STATE,
            n_jobs=1,
        )),
    ])
else:
    print("SMOTE non disponible : installer imbalanced-learn pour executer les essais SMOTE.")

resultats_binaires = []
fitted_binaires = {}

for nom_modele, pipeline in modeles_binaires.items():
    pipeline.fit(x_train_bin, y_train_bin)
    y_pred_bin = pipeline.predict(x_test_bin)

    if hasattr(pipeline, "predict_proba"):
        y_score_bin = pipeline.predict_proba(x_test_bin)[:, 1]
        auc_bin = roc_auc_score(y_test_bin, y_score_bin)
    else:
        auc_bin = np.nan

    resultats_binaires.append({
        "modele": nom_modele,
        "accuracy": accuracy_score(y_test_bin, y_pred_bin),
        "balanced_accuracy": balanced_accuracy_score(y_test_bin, y_pred_bin),
        "precision_1": precision_score(y_test_bin, y_pred_bin, zero_division=0),
        "recall_1": recall_score(y_test_bin, y_pred_bin, zero_division=0),
        "f1_1": f1_score(y_test_bin, y_pred_bin, zero_division=0),
        "roc_auc": auc_bin,
        "taux_prediction_1": y_pred_bin.mean(),
    })
    fitted_binaires[nom_modele] = pipeline

resultats_binaires = (
    pd.DataFrame(resultats_binaires)
    .sort_values(["f1_1", "balanced_accuracy", "roc_auc"], ascending=False)
    .reset_index(drop=True)
)

display(resultats_binaires.round(4))

meilleur_modele_binaire = resultats_binaires.iloc[0]["modele"]
meilleur_pipeline_binaire = fitted_binaires[meilleur_modele_binaire]
y_pred_best_bin = meilleur_pipeline_binaire.predict(x_test_bin)

matrice_confusion_binaire = pd.DataFrame(
    confusion_matrix(y_test_bin, y_pred_best_bin),
    index=["reel_0_non_satisfait", "reel_1_satisfait"],
    columns=["predit_0_non_satisfait", "predit_1_satisfait"],
)

print(f"Meilleur modele binaire selon F1 de la classe satisfaite : {meilleur_modele_binaire}")
display(matrice_confusion_binaire)

cv_binaire = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
cv_scores_binaires = cross_validate(
    meilleur_pipeline_binaire,
    x_pre,
    y_binary,
    cv=cv_binaire,
    scoring={
        "roc_auc": "roc_auc",
        "f1": "f1",
        "balanced_accuracy": "balanced_accuracy",
        "recall": "recall",
        "precision": "precision",
    },
    n_jobs=1,
)

resume_cv_binaire = pd.DataFrame({
    "metrique": ["roc_auc", "f1", "balanced_accuracy", "recall", "precision"],
    "moyenne_cv": [
        cv_scores_binaires["test_roc_auc"].mean(),
        cv_scores_binaires["test_f1"].mean(),
        cv_scores_binaires["test_balanced_accuracy"].mean(),
        cv_scores_binaires["test_recall"].mean(),
        cv_scores_binaires["test_precision"].mean(),
    ],
    "ecart_type_cv": [
        cv_scores_binaires["test_roc_auc"].std(),
        cv_scores_binaires["test_f1"].std(),
        cv_scores_binaires["test_balanced_accuracy"].std(),
        cv_scores_binaires["test_recall"].std(),
        cv_scores_binaires["test_precision"].std(),
    ],
})

print("Validation croisee du meilleur modele binaire")
display(resume_cv_binaire.round(4))


### 15.1 Interpretation attendue de l'experience binaire

Cette experience doit etre lue avec trois indicateurs principaux :

- `f1_1` : indicateur prioritaire ici, car il mesure l'equilibre entre precision et rappel sur la classe `satisfaction haute` ;
- `recall_1` : capacite a retrouver les clients reellement satisfaits ;
- `roc_auc` : capacite globale a separer les sejours satisfaits des autres ; `0.50` correspond au hasard.

L'equilibrage des classes peut se faire de deux manieres :

- `class_weight="balanced"` est la methode la plus sure pour un premier essai, car elle ne cree pas de nouvelles lignes ;
- `SMOTE` peut ameliorer le rappel de la classe minoritaire, mais il doit rester dans le pipeline pour etre applique uniquement au train.

Si l'AUC reste proche de `0.50` a `0.60`, la conclusion reste que les variables disponibles avant voyage contiennent peu de signal. Si l'AUC depasse nettement `0.70` avec une validation croisee stable, la classification binaire devient une piste plus pertinente que la regression.


### 15.2 Optimisation Random Forest avec Optuna

Cette experience reprend l'idee testee dans le notebook d'Amine : optimiser automatiquement les hyperparametres d'un `RandomForestClassifier` sur la cible binaire.

L'objectif d'Optuna est de chercher une combinaison plus pertinente de parametres : nombre d'arbres, profondeur maximale, taille minimale des feuilles, nombre minimal d'exemples pour diviser un noeud et nombre de variables testees a chaque separation.

L'optimisation se fait uniquement sur le jeu d'entrainement avec validation croisee stratifiee. Le jeu de test reste reserve a l'evaluation finale, afin d'eviter une fuite de donnees.


In [ ]:
try:
    import optuna
    optuna.logging.set_verbosity(optuna.logging.WARNING)
    optuna_disponible = True
except ImportError:
    optuna_disponible = False

if not optuna_disponible:
    print("Optuna non disponible : installer optuna pour executer cette optimisation.")
else:
    cv_optuna_binary = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

    def objective_rf_binary(trial):
        params = {
            "n_estimators": trial.suggest_int("n_estimators", 50, 300),
            "max_depth": trial.suggest_int("max_depth", 2, 15),
            "min_samples_leaf": trial.suggest_int("min_samples_leaf", 5, 30),
            "min_samples_split": trial.suggest_int("min_samples_split", 10, 80),
            "max_features": trial.suggest_categorical("max_features", ["sqrt", "log2"]),
            "class_weight": "balanced",
            "random_state": RANDOM_STATE,
            "n_jobs": 1,
        }

        pipeline = Pipeline(steps=[
            ("preprocess", clone(preprocess_binary)),
            ("model", RandomForestClassifier(**params)),
        ])

        scores = cross_validate(
            pipeline,
            x_train_bin,
            y_train_bin,
            cv=cv_optuna_binary,
            scoring="roc_auc",
            n_jobs=1,
        )
        return scores["test_score"].mean()

    study_rf_binary = optuna.create_study(
        direction="maximize",
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    )
    study_rf_binary.optimize(objective_rf_binary, n_trials=30, show_progress_bar=False)

    best_params_rf_binary = {
        **study_rf_binary.best_params,
        "class_weight": "balanced",
        "random_state": RANDOM_STATE,
        "n_jobs": 1,
    }

    pipeline_rf_optuna_binary = Pipeline(steps=[
        ("preprocess", clone(preprocess_binary)),
        ("model", RandomForestClassifier(**best_params_rf_binary)),
    ])

    pipeline_rf_optuna_binary.fit(x_train_bin, y_train_bin)
    y_pred_rf_optuna = pipeline_rf_optuna_binary.predict(x_test_bin)
    y_score_rf_optuna = pipeline_rf_optuna_binary.predict_proba(x_test_bin)[:, 1]

    resultats_rf_optuna_binary = pd.DataFrame([
        {
            "modele": "RandomForest_Optuna_binary",
            "best_cv_auc_optuna": study_rf_binary.best_value,
            "accuracy": accuracy_score(y_test_bin, y_pred_rf_optuna),
            "balanced_accuracy": balanced_accuracy_score(y_test_bin, y_pred_rf_optuna),
            "precision_1": precision_score(y_test_bin, y_pred_rf_optuna, zero_division=0),
            "recall_1": recall_score(y_test_bin, y_pred_rf_optuna, zero_division=0),
            "f1_1": f1_score(y_test_bin, y_pred_rf_optuna, zero_division=0),
            "roc_auc": roc_auc_score(y_test_bin, y_score_rf_optuna),
            "taux_prediction_1": y_pred_rf_optuna.mean(),
        }
    ])

    print("Meilleurs hyperparametres Optuna")
    display(pd.DataFrame([best_params_rf_binary]))

    print("Resultats Random Forest optimise avec Optuna")
    display(resultats_rf_optuna_binary.round(4))

    matrice_confusion_rf_optuna = pd.DataFrame(
        confusion_matrix(y_test_bin, y_pred_rf_optuna),
        index=["reel_0_non_satisfait", "reel_1_satisfait"],
        columns=["predit_0_non_satisfait", "predit_1_satisfait"],
    )
    display(matrice_confusion_rf_optuna)

    comparaison_rf_optuna = pd.concat([
        resultats_binaires[resultats_binaires["modele"].isin([
            "RandomForest_balanced",
            "RandomForest_SMOTE",
            "LogisticRegression_SMOTE",
        ])],
        resultats_rf_optuna_binary.drop(columns=["best_cv_auc_optuna"]),
    ], ignore_index=True).sort_values(["f1_1", "balanced_accuracy", "roc_auc"], ascending=False)

    print("Comparaison avec les meilleurs essais binaires precedents")
    display(comparaison_rf_optuna.round(4))


#### Interpretation de l'optimisation Optuna

Cette experience permet de verifier si le probleme vient du reglage du `RandomForest` ou du signal disponible dans les donnees.

Si Optuna n'ameliore que faiblement l'AUC ou le F1, la conclusion est que les hyperparametres ne sont pas la cause principale des faibles performances. Le modele reste limite par les variables pre-voyage disponibles dans le dataset.

Si Optuna ameliore nettement les scores, le `RandomForest` optimise pourra etre compare au modele de regression actuel et documente comme piste candidate pour la version rattrapage.


### 15.3 Test de plusieurs seuils de classification

Par defaut, un modele de classification predit la classe `1` si la probabilite est superieure ou egale a `0.50`.

Dans un contexte metier, ce seuil peut etre ajuste :

- seuil plus bas : le modele detecte plus de clients potentiellement satisfaits, mais genere plus de faux positifs ;
- seuil plus haut : le modele est plus selectif, mais rate davantage de clients satisfaits.

Cette section teste plusieurs seuils afin d'identifier le meilleur compromis entre precision, recall et F1-score.


In [ ]:
# Choix du modele de reference pour tester les seuils.
# On privilegie le modele ayant donne le meilleur F1 sur la classe satisfaite.
modele_reference_seuil = resultats_binaires.iloc[0]["modele"]
pipeline_reference_seuil = fitted_binaires[modele_reference_seuil]

# Probabilite d'appartenir a la classe 1 = satisfaction haute.
y_score_seuil = pipeline_reference_seuil.predict_proba(x_test_bin)[:, 1]

seuils = np.arange(0.10, 0.91, 0.05)
resultats_seuils = []

for seuil in seuils:
    y_pred_seuil = (y_score_seuil >= seuil).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_test_bin, y_pred_seuil).ravel()

    resultats_seuils.append({
        "modele": modele_reference_seuil,
        "seuil": seuil,
        "accuracy": accuracy_score(y_test_bin, y_pred_seuil),
        "balanced_accuracy": balanced_accuracy_score(y_test_bin, y_pred_seuil),
        "precision_1": precision_score(y_test_bin, y_pred_seuil, zero_division=0),
        "recall_1": recall_score(y_test_bin, y_pred_seuil, zero_division=0),
        "f1_1": f1_score(y_test_bin, y_pred_seuil, zero_division=0),
        "taux_prediction_1": y_pred_seuil.mean(),
        "vrais_positifs": tp,
        "faux_positifs": fp,
        "faux_negatifs": fn,
        "vrais_negatifs": tn,
    })

resultats_seuils = pd.DataFrame(resultats_seuils)

# Eviter les seuils degeneres qui predisent presque toutes les lignes en classe 1
# ou presque aucune ligne en classe 1.
seuils_exploitables = resultats_seuils[
    resultats_seuils["taux_prediction_1"].between(0.20, 0.80)
].copy()

meilleurs_seuils_f1 = (
    seuils_exploitables
    .sort_values(["f1_1", "balanced_accuracy", "precision_1"], ascending=False)
    .head(10)
    .reset_index(drop=True)
)

comparaison_seuil_05 = resultats_seuils[
    np.isclose(resultats_seuils["seuil"], 0.50)
].copy()

print(f"Modele teste : {modele_reference_seuil}")
print("Reference au seuil standard 0.50")
display(comparaison_seuil_05.round(4))
print("Top 10 des seuils exploitables selon F1 de la classe satisfaite")
display(meilleurs_seuils_f1.round(4))

seuil_optimal_f1 = float(meilleurs_seuils_f1.iloc[0]["seuil"])
y_pred_seuil_optimal = (y_score_seuil >= seuil_optimal_f1).astype(int)

matrice_confusion_seuil_optimal = pd.DataFrame(
    confusion_matrix(y_test_bin, y_pred_seuil_optimal),
    index=["reel_0_non_satisfait", "reel_1_satisfait"],
    columns=["predit_0_non_satisfait", "predit_1_satisfait"],
)

print(f"Seuil optimal selon F1 : {seuil_optimal_f1:.2f}")
display(matrice_confusion_seuil_optimal)

plt.figure(figsize=(10, 5))
plt.plot(resultats_seuils["seuil"], resultats_seuils["precision_1"], marker="o", label="Precision classe 1")
plt.plot(resultats_seuils["seuil"], resultats_seuils["recall_1"], marker="o", label="Recall classe 1")
plt.plot(resultats_seuils["seuil"], resultats_seuils["f1_1"], marker="o", label="F1 classe 1")
plt.axvline(seuil_optimal_f1, linestyle="--", color="black", label=f"Seuil optimal F1 = {seuil_optimal_f1:.2f}")
plt.xlabel("Seuil de classification")
plt.ylabel("Score")
plt.title("Impact du seuil sur precision, recall et F1")
plt.legend()
plt.grid(alpha=0.3)
plt.show()


#### Interpretation du test de seuils

Le seuil `0.50` n'est pas obligatoire : il correspond seulement au choix par defaut.

Si le meilleur seuil est inferieur a `0.50`, cela signifie que le modele produit des probabilites faibles et qu'il faut abaisser le seuil pour detecter davantage de clients satisfaits. Cette decision augmente le recall, mais elle augmente aussi le nombre de faux positifs.

Si le meilleur seuil est superieur a `0.50`, cela signifie que l'on souhaite un modele plus prudent, avec moins de predictions positives mais potentiellement une meilleure precision.

Dans ce projet, le seuil doit etre choisi selon l'objectif metier : detecter un potentiel commercial, prioriser une revue conseiller, ou eviter les fausses alertes.


## 16. Dernières opérations rattrapage — dataset v2.2 et modèle binaire optimisé

Cette section centralise les dernières opérations réalisées hors notebook sous forme de scripts reproductibles.

Objectif : tester si un dataset plus structuré, enrichi et partiellement synthétique permet d'obtenir un modèle plus exploitable que le dataset initial.

Les scripts associés sont :

- `scripts/build_signal_enrichment_dataset.py` : création de `v2.2`, augmentation à `3000` lignes, variantes `light` et `minimal` ;
- `scripts/optimize_minimal_balanced_models.py` : optimisation des hyperparamètres avec `RandomizedSearchCV` ;
- `scripts/diagnose_overfitting_minimal_balanced.py` : diagnostic overfitting train/test et validation croisée.

Attention : cette partie reste expérimentale, car une partie des lignes de `v2.2` est synthétique.


In [ ]:
import json
from pathlib import Path

import pandas as pd

PROJECT_ROOT_RATTRAPAGE = Path.cwd().resolve()
if PROJECT_ROOT_RATTRAPAGE.name == "notebooks":
    PROJECT_ROOT_RATTRAPAGE = PROJECT_ROOT_RATTRAPAGE.parent

V22_DIR = PROJECT_ROOT_RATTRAPAGE / "data" / "versions" / "v2_2_signal_enrichment"

def charger_json(path):
    path = Path(path)
    with path.open("r", encoding="utf-8") as fichier:
        return json.load(fichier)

paths_rattrapage = {
    "v22_full": V22_DIR / "signal_enrichment_report.json",
    "v22_light": V22_DIR / "light_features_experiment_report.json",
    "v22_minimal": V22_DIR / "minimal_features_experiment_report.json",
    "v22_minimal_balanced": V22_DIR / "minimal_balanced_classification_report.json",
    "v22_hyperopt": V22_DIR / "hyperparameter_optimization_report.json",
    "v22_overfitting": V22_DIR / "overfitting_diagnostic_report.json",
}

rapports_rattrapage = {
    nom: charger_json(path)
    for nom, path in paths_rattrapage.items()
}

print("Rapports chargés :")
for nom, path in paths_rattrapage.items():
    print(f"- {nom}: {path.relative_to(PROJECT_ROOT_RATTRAPAGE)}")


### 16.1 Création du dataset `v2.2` à 3000 lignes

Le dataset `v2.2` part de `1378` lignes nettoyées et enrichies, puis ajoute `1622` lignes synthétiques pour atteindre `3000` lignes.

La génération synthétique repose sur :

- un échantillonnage avec remise des lignes nettoyées ;
- une perturbation contrôlée des variables pré-voyage (`budget_total`, `prix_vol`, `duree_jours`, hébergement, météo, activité) ;
- un recalcul des scores métier pré-voyage ;
- une cible synthétique uniquement pour les nouvelles lignes, calculée à partir des règles métier pré-voyage.

Cette étape ne remplace pas une collecte réelle : elle sert à tester l'hypothèse selon laquelle un meilleur signal métier améliore le modèle.


In [ ]:
rapport_v22 = rapports_rattrapage["v22_full"]
rapport_light = rapports_rattrapage["v22_light"]
rapport_minimal = rapports_rattrapage["v22_minimal"]

resume_datasets_v22 = pd.DataFrame([
    {
        "version": "v2.2_complete",
        "lignes": rapport_v22["rows"],
        "colonnes": rapport_v22["columns_after"],
        "lignes_source": rapport_v22["original_rows"],
        "lignes_synthetiques": rapport_v22["synthetic_rows_added"],
    },
    {
        "version": "v2.2_light",
        "lignes": rapport_light["rows"],
        "colonnes": rapport_light["columns"],
        "lignes_source": rapport_v22["original_rows"],
        "lignes_synthetiques": rapport_v22["synthetic_rows_added"],
    },
    {
        "version": "v2.2_minimal",
        "lignes": rapport_minimal["rows"],
        "colonnes": rapport_minimal["columns"],
        "lignes_source": rapport_v22["original_rows"],
        "lignes_synthetiques": rapport_v22["synthetic_rows_added"],
    },
])

display(resume_datasets_v22)


### 16.2 Allègement progressif des colonnes

Trois variantes ont été testées :

- `v2.2_complete` : version complète avec toutes les colonnes d'enrichissement et de traçabilité ;
- `v2.2_light` : suppression des colonnes techniques, de traçabilité et de certaines variables intermédiaires ;
- `v2.2_minimal` : conservation des colonnes indispensables au pipeline et de quelques scores métier synthétiques.

La version `minimal` est plus lisible pour le jury et plus facile à défendre métier, tout en conservant des performances très proches de la version `light`.


In [ ]:
comparaison_versions_v22 = pd.DataFrame([
    {
        "version": "v2.2_complete",
        "meilleur_modele_regression": rapport_v22["comparison"]["best_regression_after"]["modele"],
        "MAE": rapport_v22["comparison"]["best_regression_after"]["mae"],
        "R2": rapport_v22["comparison"]["best_regression_after"]["r2"],
        "meilleur_modele_binaire": rapport_v22["comparison"]["best_binary_after"]["modele"],
        "F1_binaire": rapport_v22["comparison"]["best_binary_after"]["f1_1"],
        "ROC_AUC": rapport_v22["comparison"]["best_binary_after"]["roc_auc"],
    },
    {
        "version": "v2.2_light",
        "meilleur_modele_regression": rapport_light["best_regression_light"]["modele"],
        "MAE": rapport_light["best_regression_light"]["mae"],
        "R2": rapport_light["best_regression_light"]["r2"],
        "meilleur_modele_binaire": rapport_light["best_binary_light"]["modele"],
        "F1_binaire": rapport_light["best_binary_light"]["f1_1"],
        "ROC_AUC": rapport_light["best_binary_light"]["roc_auc"],
    },
    {
        "version": "v2.2_minimal",
        "meilleur_modele_regression": rapport_minimal["best_regression_minimal"]["modele"],
        "MAE": rapport_minimal["best_regression_minimal"]["mae"],
        "R2": rapport_minimal["best_regression_minimal"]["r2"],
        "meilleur_modele_binaire": rapport_minimal["best_binary_minimal"]["modele"],
        "F1_binaire": rapport_minimal["best_binary_minimal"]["f1_1"],
        "ROC_AUC": rapport_minimal["best_binary_minimal"]["roc_auc"],
    },
])

display(comparaison_versions_v22)


### 16.3 Distribution de la satisfaction dans `v2.2-minimal`

La version `v2.2-minimal` garde `3000` lignes et `24` colonnes.

La distribution exacte de la cible reste imparfaite mais acceptable : la classe positive binaire (`satisfaction 4-5`) représente environ un tiers des observations.

L'équilibrage retenu ne modifie pas le dataset : il est appliqué uniquement pendant l'entraînement avec `class_weight="balanced"`.


In [ ]:
dataset_minimal_path = PROJECT_ROOT_RATTRAPAGE / "data" / "versions" / "v2_2_signal_enrichment" / "travel_planning_dataset_v2_2_minimal.csv"
df_v22_minimal = pd.read_csv(dataset_minimal_path)

distribution_5_classes = (
    df_v22_minimal["satisfaction_client"]
    .value_counts()
    .sort_index()
    .rename_axis("satisfaction_client")
    .reset_index(name="nombre")
)
distribution_5_classes["pourcentage"] = (
    distribution_5_classes["nombre"] / len(df_v22_minimal) * 100
).round(2)

display(distribution_5_classes)

y_binaire_v22 = (df_v22_minimal["satisfaction_client"] >= 4).astype(int)
distribution_binaire = (
    y_binaire_v22
    .value_counts()
    .sort_index()
    .rename_axis("classe_binaire")
    .reset_index(name="nombre")
)
distribution_binaire["libelle"] = distribution_binaire["classe_binaire"].map({
    0: "satisfaction 1-3",
    1: "satisfaction 4-5",
})
distribution_binaire["pourcentage"] = (
    distribution_binaire["nombre"] / len(df_v22_minimal) * 100
).round(2)

display(distribution_binaire[["classe_binaire", "libelle", "nombre", "pourcentage"]])


### 16.4 Classification binaire avec `class_weight="balanced"`

L'objectif final testé ici est :

- `0` : satisfaction faible ou moyenne (`1`, `2`, `3`) ;
- `1` : satisfaction haute (`4`, `5`).

La pondération des classes permet de compenser le déséquilibre sans créer de nouvelles lignes supplémentaires.

Le modèle retenu à ce stade est `LogisticRegression_balanced`, car il obtient le meilleur compromis sur `F1`, `recall` et `balanced accuracy`.


In [ ]:
rapport_balanced = rapports_rattrapage["v22_minimal_balanced"]
resultats_balanced = pd.DataFrame(rapport_balanced["results"])

display(resultats_balanced[[
    "modele",
    "class_weight",
    "accuracy",
    "balanced_accuracy",
    "precision_1",
    "recall_1",
    "f1_1",
    "roc_auc",
]])

print("Meilleur modèle équilibré :")
display(pd.DataFrame([rapport_balanced["best_model"]]))


### 16.5 Optimisation des hyperparamètres avec `RandomizedSearchCV`

Deux familles de modèles ont été optimisées :

- `LogisticRegression` : recherche sur `C`, `penalty`, `solver` ;
- `RandomForest` : recherche sur `n_estimators`, `max_depth`, `min_samples_leaf`, `min_samples_split`, `max_features`, `class_weight`.

La validation croisée est stratifiée en `5 folds` avec `F1` comme métrique d'optimisation, car la classe positive reste minoritaire.


In [ ]:
rapport_hyperopt = rapports_rattrapage["v22_hyperopt"]
resultats_hyperopt = pd.DataFrame(rapport_hyperopt["all_results_sorted"])
colonnes_hyperopt = [
    "modele",
    "accuracy",
    "balanced_accuracy",
    "precision_1",
    "recall_1",
    "f1_1",
    "roc_auc",
    "cv_best_f1",
]

display(resultats_hyperopt[[col for col in colonnes_hyperopt if col in resultats_hyperopt.columns]])

print("Gains vs meilleur modèle initial :")
display(pd.DataFrame([rapport_hyperopt["gains_vs_best_initial"]]))

print("Meilleurs hyperparamètres du modèle retenu :")
best_model_hyperopt = rapport_hyperopt["best_model_after_optimization"]
display(pd.DataFrame([{
    "modele": best_model_hyperopt["modele"],
    "best_params": best_model_hyperopt.get("best_params", {}),
}]))


### 16.6 Diagnostic overfitting du modèle optimisé

Le meilleur modèle après optimisation est `LogisticRegression_balanced_optimized`.

Le diagnostic compare :

- les performances sur le jeu d'entraînement ;
- les performances sur le jeu de test ;
- les écarts moyens en validation croisée.

L'objectif est de vérifier si le modèle mémorise le train ou s'il généralise correctement.


In [ ]:
rapport_overfitting = rapports_rattrapage["v22_overfitting"]

print("Métriques train/test :")
display(pd.DataFrame(rapport_overfitting["holdout_metrics"]))

print("Écarts train/test :")
display(pd.DataFrame([rapport_overfitting["gaps"]]))

print("Validation croisée - synthèse :")
cv_rows = []
for metrique, valeurs in rapport_overfitting["cross_validation_summary"].items():
    cv_rows.append({
        "metrique": metrique,
        "moyenne": valeurs["mean"],
        "ecart_type": valeurs["std"],
    })
display(pd.DataFrame(cv_rows))

print("Conclusion :")
print(rapport_overfitting["conclusion"])


### 16.7 Décision actualisée pour la version rattrapage

Au regard des dernières expériences, la meilleure piste n'est plus la régression sur la note exacte, mais la **classification binaire pré-voyage**.

Modèle candidat retenu : `LogisticRegression_balanced_optimized`.

Résultats clés :

- `F1 = 0.5995` ;
- `ROC AUC = 0.7320` ;
- `balanced accuracy = 0.7057` ;
- pas de signe fort d'overfitting (`écart F1 train/test = 0.0236`).

Interprétation métier : le modèle devient exploitable comme prototype pour estimer un risque de satisfaction haute ou non, avant le départ.

Réserve importante : la version `v2.2-minimal` contient des lignes synthétiques. Le modèle peut donc être industrialisé comme prototype contrôlé, mais il doit être validé sur des données réelles avant toute mise en production décisionnelle.
